### Bivariate test on Adult dataset

In [10]:
# Minimal: run bivariate_test for real vs tabsyn
import os
from tabeva.metrics.bivariate import bivariate_test, bivariate_plot
import pandas as pd

os.makedirs('output/bivariate', exist_ok=True)
real = pd.read_csv('synthetic/adult/real.csv')
fake = pd.read_csv('synthetic/adult/tabsyn.csv')
# auto-detect categorical columns (override if needed)
c_col = [c for c in real.columns if real[c].dtype == 'object' or real[c].nunique() <= 20]

delta_bi, abs_diff = bivariate_test(real, fake, c_col)
print('Average absolute bivariate discrepancy (real vs tabsyn):', delta_bi)
abs_diff.to_csv('output/bivariate/abs_diff_real_tabsyn.csv')
bivariate_plot(abs_diff, filename='output/bivariate/abs_diff_real_tabsyn')
print('Saved abs_diff and plot to output/bivariate/')


TypeError: ufunc 'divide' not supported for the input types, and the inputs could not be safely coerced to any supported types according to the casting rule ''safe''

In [ ]:
# Run bivariate_test for all synthetic models in synthetic/adult
import os
import glob
import pandas as pd
from tabeva.preprocessing import data_preprocess
from tabeva.metrics.bivariate import bivariate_test, bivariate_plot

os.makedirs('output/bivariate', exist_ok=True)
real = pd.read_csv('adult/real.csv')

# Detect categorical columns using TABEVA heuristic (override if needed)
cat_cols = [c for c in real.columns if real[c].dtype == 'object' or real[c].nunique() <= 20]
print('Detected categorical columns:', cat_cols)

syn_dir = os.path.join('synthetic', 'adult')
files = sorted(glob.glob(os.path.join(syn_dir, '*.csv')))
results = []

for path in files:
    name = os.path.splitext(os.path.basename(path))[0]
    if name in ('real', 'test'):
        continue
    print('\nProcessing', name)
    try:
        fake = pd.read_csv(path)
    except Exception as e:
        print(f'  Failed to read {path}: {e}')
        continue

    try:
        # Preprocess (label-encode cats, scale nums)
        real_p, fake_p = data_preprocess(real, fake, cat_cols, onehot=False)
        delta, abs_diff = bivariate_test(real_p, fake_p, cat_cols)
        results.append({'method': name, 'delta_bi': float(delta)})

        # Save per-method outputs
        out_csv = os.path.join('output', 'bivariate', f'{name}_abs_diff.csv')
        abs_diff.to_csv(out_csv)
        try:
            bivariate_plot(abs_diff, filename=os.path.join('output', 'bivariate', f'{name}_abs_diff'))
        except Exception:
            pass
        print(f'  {name}: delta_bi={delta}')
    except Exception as e:
        print(f'  Failed processing {name}: {e}')

# Summary
res_df = pd.DataFrame(results).sort_values('delta_bi')
res_df.to_csv(os.path.join('output', 'bivariate', 'summary_bivariate_adult.csv'), index=False)
print('\nSummary saved to output/bivariate/summary_bivariate_adult.csv')
print(res_df.to_string(index=False))


Detected categorical columns: ['workclass', 'education', 'marital.status', 'occupation', 'relationship', 'race', 'sex', 'native.country', 'income']
Average absolute bivariate discrepancy (real vs tabsyn, processed): 0.04603403669798408
Saved abs_diff and plot to output/bivariate/


In [13]:
delta_bi

0.04603403669798408